# EDA
## merge des exports issus de la Data prep

Notebooks de Data prep :
- bdiff.ipynb
- communes.ipynb

Etapes de l'EDA :
- import des incendies et des communes en .parquet
- jointure
- analyse

In [ ]:
import sys
from pathlib import Path

from IPython.display import HTML, display

import chardet
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import geopandas as gpd
import folium
from folium.plugins import HeatMap, MarkerCluster
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import HTML, IFrame, display
import base64


# adds parent file of the current directory
# to the paths in which Python looks for modules to import
# in the current Python process
sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW_DIR, DATA_CLEAN_DIR, GEO_DATA_CLEAN_DIR, ROOT_DIR, FIGURES_DIR
from utils.cleaning_utils import delete, normalize_columns_names, normalize_text_columns_cells, optimize_numeric_column, fill_rate
from utils.analysis_utils import plot_missing_bar, plot_numeric_histograms, plot_corr_heatmap, plot_qualitative

In [ ]:
df_incendies = pd.read_parquet(
    DATA_CLEAN_DIR / "incendies.parquet"
)

df_communes = pd.read_parquet(
    GEO_DATA_CLEAN_DIR / "communes.parquet"
)

In [ ]:
df_incendies.head(2)

,departement,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,surfaces_non_boisees_naturelles_m2,surfaces_non_boisees_artificialisees_m2,surfaces_non_boisees_m2,type_de_peuplement,nature
0,2a,2a198,2011-01-01 00:33:00,50,24,24,0,<NA>,<NA>,<NA>,<NA>,<NA>,2,1,involontaire (particulier)
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,4,NaN


In [ ]:
df_communes.head(2)

,code_insee,localisation,nom_standard,region,departement,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude,longitude
0,01001,1400,L'Abergement-Clémenciat,84,01,832,1565,53.0,242,206,272,46.151,4.921
1,01002,1640,L'Abergement-de-Varey,84,01,267,912,29.0,483,290,748,46.007,5.423


#### Fusion des dataset incendies/BDIFF et liste des communes/data.gouv.fr
sur la clé code_insee

In [ ]:
df = df_incendies.merge(
    df_communes,
    on="code_insee",
    how="left"
)

In [ ]:
df.head(2)

,departement_x,code_insee,date_de_premiere_alerte,surface_parcourue_m2,surface_foret_m2,surface_maquis_garrigues_m2,autres_surfaces_naturelles_hors_foret_m2,surfaces_agricoles_m2,autres_surfaces_m2,surface_autres_terres_boisees_m2,...,region,departement_y,population,superficie_hectare,densite,altitude_moyenne,altitude_minimale,altitude_maximale,latitude,longitude
0,2a,2a198,2011-01-01 00:33:00,50,24,24,0,<NA>,<NA>,<NA>,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,11,11203,2011-01-02 09:10:00,1,1,0,<NA>,<NA>,<NA>,<NA>,...,76,11,10952,3815,287.0,58,19,187,43.2,2.758


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 52806 entries, 0 to 52805
Data columns (total 27 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   departement_x                             52806 non-null  str           
 1   code_insee                                52806 non-null  object        
 2   date_de_premiere_alerte                   52806 non-null  datetime64[us]
 3   surface_parcourue_m2                      52806 non-null  Int32         
 4   surface_foret_m2                          48156 non-null  Int32         
 5   surface_maquis_garrigues_m2               28073 non-null  Int32         
 6   autres_surfaces_naturelles_hors_foret_m2  34531 non-null  Int32         
 7   surfaces_agricoles_m2                     6588 non-null   Int32         
 8   autres_surfaces_m2                        6572 non-null   Int32         
 9   surface_autres_terres_boisees_m2       

In [ ]:
# Préparation des données pour la heatmap
# Chaque point contient [latitude, longitude, poids/intensité]
# heat_data = [[row['latitude'], row['longitude'], row['risk_level']]
#              for idx, row in df.iterrows()]

# Ajout de la heatmap à la carte
# HeatMap(
#     heat_data,
#     min_opacity=0.2,
#     radius=15,
#     blur=15,
#     max_zoom=1,
#     gradient={0.2: 'blue', 0.4: 'cyan', 0.6: 'lime', 0.8: 'yellow', 1.0: 'red'}
# ).add_to(m_heatmap)

In [ ]:
## Carte des incendies

# Calcul du centre de la carte
center_lat = df['latitude'].mean()
center_lon = df['longitude'].mean()

# Création de la carte de base
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=7,
    # tiles='OpenStreetMap'
    tiles="CartoDB dark_matter"
)


# Ajout d'un titre à la carte
folium.map.Marker(
    [center_lat + 0.5, center_lon],
    icon=folium.DivIcon(
        html=f'<div style="font-size: 14px; font-weight: bold;">Incendies</div>',
        icon_size=(200, 20),
        icon_anchor=(0, 0)
    )
).add_to(m)

# print(f"Heatmap créée avec {len(heat_data)} points de données")
print(f"Centre de la carte: ({center_lat:.2f}, {center_lon:.2f})")

# Affichage de la carte
m

#  Rendu explicite pour contourner le blocage JavaScript dans VS Code REND LE FOND DU NOTEBOK BLANC
# display(HTML(m.get_root().render()))

# Sauvegarde de la carte dans un fichier HTML local
# map_path = FIGURES_DIR / "carte_incendies.html"
# m.save(map_path)



Centre de la carte: (42.48, 4.45)


/home/coule/Documents/projets/incendies/.venv/lib/python3.11/site-packages/folium/raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore


In [ ]:
# Définition du chemin vers la carte enregistrée
map_path = Path(ROOT_DIR / "figures" / "carte_incendies.html")

if map_path.is_file():
    # Lecture du fichier HTML en texte brut
    raw_html_content = map_path.read_text(encoding="utf-8")

    # Encodage en base64 pour garantir une transmission sécurisée et sans coupure
    encoded_bytes = base64.b64encode(raw_html_content.encode("utf-8"))
    encoded_string = encoded_bytes.decode("utf-8")

    # Construction de l'iframe avec l'URI de données
    iframe_element = (
        f'<iframe '
        f'src="data:text/html;charset=utf-8;base64,{encoded_string}" '
        f'width="100%" '
        f'height="600" '
        f'frameborder="0">'
        f'</iframe>'
    )

    # Affichage explicite du composant HTML
    display(HTML(iframe_element))
else:
    print(f"Erreur : le fichier {map_path} est introuvable.")

/home/coule/Documents/projets/incendies/.venv/lib/python3.11/site-packages/IPython/core/display.py:448: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")
